# Background-Robust Static Hand-Gesture Recognition - Colab Pipeline

This notebook is the cloud-based execution environment for the paper *"Background-Robust Static Hand-Gesture Recognition: An Image-Grouped Multi-Backbone Ablation"*. It mirrors the local single-GPU workflow described in `README.md` and `docs/REPRODUCTION.md`, allowing module-by-module execution of the experimental stages or a single end-to-end automated run.

## Environmental Configuration
The initial setup clones the repository, mounts the required Google Drive storage for model and report backups, and stages the raw RPS dataset together with the background scenes that the synthetic pipeline composites onto.


In [ ]:
# @title Initial Setup and Dependency Installation
import os
import zipfile
from google.colab import drive

# Repository Configuration
GITHUB_REPO_URL = "https://github.com/RageLog/bg-robust-rps" # @param {type:"string"}
GITHUB_BRANCH = "main" # @param {type:"string"}

# Google Drive Integration
drive.mount('/content/drive')
DRIVE_DIR = "/content/drive/MyDrive/RPC_Colab"
DATA_ZIP = os.path.join(DRIVE_DIR, "datasets.zip")
PROJECT_DIR = "/content/rpc_project"
LOCAL_DIR = PROJECT_DIR  # git clone target == repository root

if not os.path.exists(DRIVE_DIR):
    print(f"[Warning] Please create the directory '{DRIVE_DIR}' in your Google Drive and upload 'datasets.zip'.")
else:
    # Repository cloning and updating
    if os.path.exists(PROJECT_DIR):
        print("[System] Updating repository...")
        os.chdir(PROJECT_DIR)
        !git reset --hard
        !git pull origin {GITHUB_BRANCH}
    else:
        print("[System] Cloning repository...")
        !git clone -b {GITHUB_BRANCH} {GITHUB_REPO_URL} {PROJECT_DIR}

    os.chdir(LOCAL_DIR)

    # Drive backup mirror for models/ and reports/
    os.makedirs(os.path.join(DRIVE_DIR, "models"), exist_ok=True)
    os.makedirs(os.path.join(DRIVE_DIR, "reports"), exist_ok=True)

    # Dependency management
    if os.path.exists('requirements.txt'):
        print("[System] Installing dependencies...")
        !pip install -q -r requirements.txt
        !pip install -q rembg[gpu]

    # Dataset extraction (expects datasets.zip produced by zip_datasets_for_colab.py:
    # the archive should contain a top-level 'datasets/' tree with raw/ and backgrounds/)
    raw_path = os.path.join(LOCAL_DIR, "datasets", "raw")
    if not os.path.exists(raw_path) and os.path.exists(DATA_ZIP):
        print(f"[System] Extracting dataset from {DATA_ZIP}...")
        with zipfile.ZipFile(DATA_ZIP, 'r') as zip_ref:
            zip_ref.extractall(LOCAL_DIR)
        print("[System] Dataset extraction complete.")
    else:
        print("[System] Dataset partition is already established or archive is missing.")

    print("\n[Success] Workspace initialization complete. Proceed to independent modules.")


---
## Experimental Stages (Modular Execution)
The methodology is divided into independent atomic stages. Each cell corresponds to a phase in the research methodology. Ablation tags follow the paper's taxonomy: `NO_SYNTH` (no augmentation), `REMBG_ONLY`, `RANDBG`, `INDOOR`, `FULL`, plus the secondary modes `BASELINE`, `STYLE_TRANSFER`, `GAN`, `RATIO_1X`, `NO_SHIFT`, `NO_ALPHA`.


### Stage 1: Synthetic Data Generation and Partitioning
Foreground hands are extracted with U2-Net (rembg) and composited onto various background conditions, controlled by the ablation mode. Image-grouped train/validation/test splits are then materialised under `datasets/synthetic_<ablation>/`.


In [ ]:
# @title Stage 1: Data Preparation Pipeline
import os
os.chdir(LOCAL_DIR)

ABLATION_MODE = "FULL" # @param ["FULL", "INDOOR", "RANDBG", "REMBG_ONLY", "BASELINE", "NO_SYNTH", "STYLE_TRANSFER", "GAN", "RATIO_1X", "NO_SHIFT", "NO_ALPHA"]
MODEL_NAME = "EfficientNetV2B0" # @param ["EfficientNetV2B0", "ResNet50", "MobileNetV3Small", "DenseNet121", "VGG16"]

print(f"\n[INFO] Starting Data Prep for Configuration: {ABLATION_MODE}")
if ABLATION_MODE != "NO_SYNTH":
    !python src/generate_data.py --ablation {ABLATION_MODE} --model {MODEL_NAME}

!python src/split_data.py --ablation {ABLATION_MODE}


### Stage 2 & 8: Convolutional Neural Network Training
Trains a backbone under the chosen ablation, fine-tuning protocol, and classification head. Checkpoint and per-fold JSON report file names embed `head_strategy` and `tune_strategy`, matching the schema produced by `src/train.py` (`{RUN_NAME}_{model}_{ablation}_{head}_{tune}_*best.keras`).


In [ ]:
# @title Stage 2: Deep Learning Model Training
import os
os.chdir(LOCAL_DIR)

ABLATION_MODE = "FULL" # @param ["FULL", "INDOOR", "RANDBG", "REMBG_ONLY", "BASELINE", "NO_SYNTH", "STYLE_TRANSFER", "GAN", "RATIO_1X", "NO_SHIFT", "NO_ALPHA"]
MODEL_NAME = "EfficientNetV2B0" # @param ["EfficientNetV2B0", "ResNet50", "MobileNetV3Small", "DenseNet121", "VGG16"]
TUNE_STRATEGY = "standard" # @param ["standard", "progressive"]
HEAD_STRATEGY = "standard" # @param ["standard", "spatial_pooling", "attention"]
K_FOLD = 5 # @param {type:"slider", min:1, max:10, step:1}

print(f"[INFO] Initializing Training Sequence | Architecture: {MODEL_NAME} | Ablation: {ABLATION_MODE} | Head: {HEAD_STRATEGY} | Tune: {TUNE_STRATEGY} | K-Fold: {K_FOLD}")
cmd = f"python src/train.py --ablation {ABLATION_MODE} --model {MODEL_NAME} --tune_strategy {TUNE_STRATEGY} --head_strategy {HEAD_STRATEGY}"
if K_FOLD > 1:
    cmd += f" --cv {K_FOLD}"
!{cmd}

print("[System] Attempting cloud synchronization...")
!cp -r models/. /content/drive/MyDrive/RPC_Colab/models/
!cp -r reports/. /content/drive/MyDrive/RPC_Colab/reports/


### Stage 4: Classical Machine Learning Baseline
Establishes the auxiliary MediaPipe Hands + RBF-SVM landmark baseline reported in the paper. The script trains and evaluates on the `FULL` ablation split.


In [ ]:
# @title Stage 4: SVM Baseline Execution
import os
os.chdir(LOCAL_DIR)

!python src/baseline_mediapipe_svm.py --ablation FULL
!cp -r models/. /content/drive/MyDrive/RPC_Colab/models/
!cp -r reports/. /content/drive/MyDrive/RPC_Colab/reports/


### Stage 5: Explainability Methods (Grad-CAM)
Generates Class Activation Heatmaps for the trained checkpoint that matches the selected ablation / head / tune / fold combination. The lookup uses a glob over `models/` so it tolerates either single-split (`*_best.keras`) or k-fold (`*_fold_<n>_best.keras`) artefacts.


In [ ]:
# @title Stage 5: Grad-CAM Feature Visualization
import os
import sys
import glob
os.chdir(LOCAL_DIR)

# Pull RUN_NAME from src/config.py so model file naming stays in sync.
if os.path.join(LOCAL_DIR, "src") not in sys.path:
    sys.path.append(os.path.join(LOCAL_DIR, "src"))
import config

ABLATION_MODE = "FULL" # @param ["FULL", "INDOOR", "RANDBG", "REMBG_ONLY", "BASELINE", "NO_SYNTH", "STYLE_TRANSFER", "GAN", "RATIO_1X", "NO_SHIFT", "NO_ALPHA"]
MODEL_NAME = "EfficientNetV2B0" # @param ["EfficientNetV2B0", "ResNet50", "MobileNetV3Small", "DenseNet121", "VGG16"]
HEAD_STRATEGY = "standard" # @param ["standard", "spatial_pooling", "attention"]
TUNE_STRATEGY = "standard" # @param ["standard", "progressive"]
FOLD = "any" # @param ["any", "1", "2", "3", "4", "5", "single_split"]

name_base = f"{config.RUN_NAME}_{MODEL_NAME}_{ABLATION_MODE.lower()}_{HEAD_STRATEGY}_{TUNE_STRATEGY}"
if FOLD == "single_split":
    pattern = os.path.join(config.MODELS_DIR, f"{name_base}_best.keras")
elif FOLD == "any":
    pattern = os.path.join(config.MODELS_DIR, f"{name_base}_*best.keras")
else:
    pattern = os.path.join(config.MODELS_DIR, f"{name_base}_fold_{FOLD}_best.keras")

matches = sorted(glob.glob(pattern))
if not matches:
    print(f"[Error] Weights not found for pattern: {pattern}")
else:
    model_path = matches[0]
    print(f"[INFO] Using checkpoint: {model_path}")
    test_dir = os.path.join(LOCAL_DIR, "datasets", f"synthetic_{ABLATION_MODE.lower()}", "test")
    if not os.path.exists(test_dir):
        test_dir = os.path.join(LOCAL_DIR, "datasets", f"synthetic_{ABLATION_MODE.lower()}", "val")
    out_dir = os.path.join(LOCAL_DIR, "reports", MODEL_NAME, ABLATION_MODE.upper(), "gradcam")

    !python src/gradcam_vis.py --model_path {model_path} --test_dir {test_dir} --out_dir {out_dir}

    !cp -r reports/. /content/drive/MyDrive/RPC_Colab/reports/


### Stage 7 & 11: Cross-Dataset Evaluation
Runs the held-out HaGRID slice probe used in §IV-G of the paper. `evaluate_cross_dataset.py` auto-prepares the external split via `prepare_hagrid_annotations.py` and then evaluates every checkpoint in `models/`, so this single call covers the zero-shot grid in one pass.


In [ ]:
# @title Stage 7: External Validation Testing
import os
os.chdir(LOCAL_DIR)

print("\n[INFO] Commencing Zero-Shot Evaluation across all trained checkpoints...")
!python src/evaluate_cross_dataset.py

!cp -r reports/. /content/drive/MyDrive/RPC_Colab/reports/


### Stage 9: Temporal Smoothing Optimization
Performs the deployment-side N x theta grid search reported in Table 15. Output lands in `reports/aggregated/temporal_smoothing_*.json`.


In [ ]:
# @title Stage 9: Temporal Optimization Run
import os
os.chdir(LOCAL_DIR)
print("\n[INFO] Commencing Temporal Smoothing Grid Search...")
!python src/optimize_temporal_smoothing.py

!cp -r reports/. /content/drive/MyDrive/RPC_Colab/reports/


### Stage 10: Segmentation Quality Evaluation
Quantitative evaluation of the U2-Net background removal phase, surfaced as the foreground-extraction quality probe (`reports/aggregated/rembg_*.json`).


In [ ]:
# @title Stage 10: Segmentation Metrics Extractor
import os
os.chdir(LOCAL_DIR)
print("\n[INFO] Extracting Segmentation Quality Metrics...")
!python src/evaluate_segmentation.py

!cp -r reports/. /content/drive/MyDrive/RPC_Colab/reports/


---
## Unsupervised Multi-Stage Execution Architecture
The cell below launches `run_all_experiments.py`, the auto-pilot that walks the full experimental grid (10 ablations x 5 backbones x 3 heads x 2 fine-tuning protocols within the FULL/EfficientNetV2B0 cell, plus baselines, Grad-CAM, segmentation probe, temporal grid, and cross-dataset evaluation). `--run-mode missing` skips configurations whose checkpoints already exist; `--run-mode full` retrains everything. Expect overnight runtimes on a single T4.


In [ ]:
# @title Automated Pipeline Execution
import os
os.chdir(LOCAL_DIR)

RUN_MODE = "missing" # @param ["missing", "full"]
TUNE_STRATEGY = "both" # @param ["standard", "progressive", "both"]

print(f"[System] Initiating Research Pipeline | Mode: {RUN_MODE} | Tune: {TUNE_STRATEGY}")
!python run_all_experiments.py --run-mode {RUN_MODE} --tune_strategy {TUNE_STRATEGY}


---
## Fault Tolerance Protocol
Refreshes the aggregated CSVs under `reports/aggregated/` from the per-fold JSON reports already on disk - no retraining. Run this after any partial recovery or Drive sync to keep the paper artefacts in sync with the actual measurements.


In [ ]:
# @title Aggregated Report Regeneration
import os
os.chdir(LOCAL_DIR)
print("[System] Regenerating aggregated CSV reports from per-fold JSON...")
!python regenerate_reports.py

!cp -r reports/. /content/drive/MyDrive/RPC_Colab/reports/
